# 04 SQL Analysis

This notebook uses SQL to answer business questions about Melbourne metropolitan train patronage, using the cleaned master dataset created during data preparation.

The goal is to demonstrate SQL skills including filtering, aggregation, grouping, ordering, joins-style thinking, subqueries, and window functions.

In [1]:
# Import Libraries
import pandas as pd
import sqlite3
from pathlib import Path

In [3]:
# Load Processed Dataset
PROJECT_ROOT = Path("..")

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "train_patronage_master.csv"
)

df = pd.read_csv(DATA_PATH)

df.head()

,financial_year,stop_id,stop_name,stop_lat,stop_long,pax_annual,pax_weekday,pax_norm_weekday,pax_sch_hol_weekday,pax_saturday,pax_sunday,pax_pre_am_peak,pax_am_peak,pax_interpeak,pax_pm_peak,pax_pm_late
0,2018-2019,19827,Stony Point,-38.374235,145.221837,5400,50,50,50,50,50,50,50,50,50,50
1,2018-2019,19828,Crib Point,-38.366123,145.204043,17900,50,50,50,150,100,50,50,50,50,50
2,2018-2019,19829,Morradoo,-38.354033,145.189603,5650,50,50,50,50,50,50,50,50,50,50
3,2018-2019,19830,Bittern,-38.337390,145.178026,14800,50,50,50,50,50,50,50,50,50,50
4,2018-2019,19831,Hastings,-38.305659,145.185980,21900,50,50,50,50,50,50,50,50,50,50


In [4]:
# Creating SQlite Database
DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "train_patronage.db"
)

conn = sqlite3.connect(DB_PATH)

df.to_sql(
    "train_patronage",
    conn,
    if_exists="replace",
    index=False
)

1555

In [5]:
# Helper function to run query
def run_query(query):
    """
    Execute a SQL query and return results as a pandas DataFrame.
    """
    return pd.read_sql_query(query, conn)

In [6]:
#CHeck table structure
query = """
PRAGMA table_info(train_patronage);
"""

run_query(query)

,cid,name,type,notnull,dflt_value,pk
0,0,financial_year,TEXT,0,None,0
1,1,stop_id,INTEGER,0,None,0
2,2,stop_name,TEXT,0,None,0
3,3,stop_lat,REAL,0,None,0
4,4,stop_long,REAL,0,None,0
5,5,pax_annual,INTEGER,0,None,0
6,6,pax_weekday,INTEGER,0,None,0
7,7,pax_norm_weekday,INTEGER,0,None,0
8,8,pax_sch_hol_weekday,INTEGER,0,None,0
9,9,pax_saturday,INTEGER,0,None,0


In [7]:
query_1 = "SELECT * FROM train_patronage"
run_query(query_1)

,financial_year,stop_id,stop_name,stop_lat,stop_long,pax_annual,pax_weekday,pax_norm_weekday,pax_sch_hol_weekday,pax_saturday,pax_sunday,pax_pre_am_peak,pax_am_peak,pax_interpeak,pax_pm_peak,pax_pm_late
0,2018-2019,19827,Stony Point,-38.374235,145.221837,5400,50,50,50,50,50,50,50,50,50,50
1,2018-2019,19828,Crib Point,-38.366123,145.204043,17900,50,50,50,150,100,50,50,50,50,50
2,2018-2019,19829,Morradoo,-38.354033,145.189603,5650,50,50,50,50,50,50,50,50,50,50
3,2018-2019,19830,Bittern,-38.337390,145.178026,14800,50,50,50,50,50,50,50,50,50,50
4,2018-2019,19831,Hastings,-38.305659,145.185980,21900,50,50,50,50,50,50,50,50,50,50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1550,2024-2025,49458,East Pakenham,-38.084284,145.506311,54500,150,150,100,200,200,50,50,50,50,50
1551,2024-2025,52095,Southland,-37.958758,145.049124,479000,1350,1300,1500,1400,1100,50,250,450,500,100
1552,2024-2025,52159,Middle Gorge,-37.644062,145.092140,222950,750,850,500,300,200,100,350,100,250,50
1553,2024-2025,52160,Hawkstowe,-37.622992,145.097396,208500,700,750,550,350,200,150,400,100,50,50


In [10]:
# Total network patronage by financial year

query_2 = """
SELECT 
    financial_year,
    SUM(pax_annual) as total_patronage
    FROM train_patronage
    GROUP BY financial_year
    ORDER BY financial_year;
"""

run_query(query_2)

,financial_year,total_patronage
0,2018-2019,231607950
1,2019-2020,178626800
2,2020-2021,77820000
3,2021-2022,94736800
4,2022-2023,149195750
5,2023-2024,173839400
6,2024-2025,178480700


In [11]:
# Top 10 busiest stations across all years

query_3 = """
SELECT 
    stop_name,
    SUM(pax_annual) as total_patronage
    FROM train_patronage
    GROUP BY stop_name
    ORDER BY total_patronage DESC
    LIMIT 10;
"""

run_query(query_3)

,stop_name,total_patronage
0,Flinders Street,128516900
1,Southern Cross,84848350
2,Melbourne Central,67494150
3,Parliament,39826900
4,Footscray,26917450
5,Flagstaff,21436800
6,Richmond,20992950
7,South Yarra,20254550
8,Caulfield,19056250
9,Box Hill,16148500


In [12]:
# Bottom 10 lowest-use stations
query_4 = """
SELECT 
    stop_name,
    SUM(pax_annual) as total_patronage
    FROM train_patronage
    GROUP BY stop_name
    ORDER BY total_patronage ASC
    LIMIT 10;
"""

run_query(query_4)

,stop_name,total_patronage
0,Baxter,15950
1,Morradoo,24300
2,Tyabb,25050
3,Stony Point,36400
4,Leawarra,41600
5,Somerville,54250
6,Bittern,57850
7,East Pakenham,58200
8,Crib Point,68350
9,Hastings,93200


In [14]:
# Top 10 stations in the latest year
query_5 = """
SELECT 
    stop_name,
    pax_annual as total_patronage
    FROM train_patronage
    WHERE financial_year = '2024-2025'
    ORDER BY total_patronage DESC
    LIMIT 10;
"""

run_query(query_5)

,stop_name,total_patronage
0,Flinders Street,19633300
1,Southern Cross,14696050
2,Melbourne Central,11876900
3,Parliament,6344650
4,Footscray,4383950
5,Richmond,3712750
6,Flagstaff,3445200
7,Caulfield,3205100
8,South Yarra,3199050
9,Box Hill,2648350


In [15]:
# COVID impact by station
query_6 = """
SELECT
    pre.stop_name,
    pre.pax_annual AS pre_covid_patronage,
    covid.pax_annual AS covid_patronage,
    covid.pax_annual - pre.pax_annual AS patronage_change,
    ROUND(
        (covid.pax_annual - pre.pax_annual) * 100.0
        / pre.pax_annual,
        2
    ) AS patronage_change_pct
FROM train_patronage AS pre
JOIN train_patronage AS covid
    ON pre.stop_name = covid.stop_name
WHERE pre.financial_year = '2018-2019'
  AND covid.financial_year = '2020-2021'
ORDER BY patronage_change_pct ASC
LIMIT 10;
"""

run_query(query_6)

,stop_name,pre_covid_patronage,covid_patronage,patronage_change,patronage_change_pct
0,Showgrounds,180000,4150,-175850,-97.69
1,Flemington Racecourse,212350,25350,-187000,-88.06
2,Coburg,728500,145500,-583000,-80.03
3,Parliament,10124200,2213650,-7910550,-78.14
4,Aircraft,303150,70150,-233000,-76.86
5,Southern Cross,19551450,4556450,-14995000,-76.70
6,Williams Landing,1756650,411450,-1345200,-76.58
7,Moreland,380850,91250,-289600,-76.04
8,East Malvern,481150,127600,-353550,-73.48
9,Diggers Rest,221250,59350,-161900,-73.18


In [19]:
# Peak-period demand by station
query_7 = """
SELECT 
    stop_name,
    SUM(pax_am_peak + pax_pm_peak) AS total_peak_patronage
FROM train_patronage
GROUP BY stop_name
ORDER BY total_peak_patronage DESC
LIMIT 10;
"""

run_query(query_7)

,stop_name,total_peak_patronage
0,Flinders Street,249100
1,Southern Cross,206150
2,Melbourne Central,137950
3,Parliament,111700
4,Flagstaff,61200
5,Footscray,55500
6,South Yarra,46050
7,Caulfield,35650
8,Box Hill,33250
9,Richmond,31200


In [27]:
# Weekend share by financial year

query_8 = """
SELECT
    financial_year,
    SUM(pax_weekday) AS total_weekday_patronage,
    SUM(pax_saturday + pax_sunday) AS total_weekend_patronage,
    ROUND(
        SUM(pax_saturday + pax_sunday) * 100.0
        /(SUM(pax_weekday) + SUM(pax_saturday + pax_sunday)),2
    ) AS weekend_share_pct
FROM train_patronage
GROUP BY financial_year
ORDER BY financial_year;
"""

run_query(query_8)

,financial_year,total_weekday_patronage,total_weekend_patronage,weekend_share_pct
0,2018-2019,779700,660150,45.85
1,2019-2020,597650,513600,46.22
2,2020-2021,246850,274150,52.62
3,2021-2022,296250,359150,54.80
4,2022-2023,477350,538350,53.00
5,2023-2024,560200,593800,51.46
6,2024-2025,570400,594500,51.03


In [30]:
# Ranking stations by annual patronage within each year
query_9 = """
WITH ranked_stations AS (

    SELECT
        financial_year,
        stop_name,
        pax_annual,

        RANK() OVER (
            PARTITION BY financial_year
            ORDER BY pax_annual DESC
        ) AS station_rank

    FROM train_patronage

)

SELECT *
FROM ranked_stations
WHERE station_rank <= 3
ORDER BY financial_year, station_rank;
"""

run_query(query_9)

,financial_year,stop_name,pax_annual,station_rank
0,2018-2019,Flinders Street,28320650,1
1,2018-2019,Southern Cross,19551450,2
2,2018-2019,Melbourne Central,15250700,3
3,2019-2020,Flinders Street,21503500,1
4,2019-2020,Southern Cross,14895400,2
5,2019-2020,Melbourne Central,10865600,3
6,2020-2021,Flinders Street,8528000,1
7,2020-2021,Southern Cross,4556450,2
8,2020-2021,Melbourne Central,4206700,3
9,2021-2022,Flinders Street,11392850,1


### Insight

Flinders Street, Southern Cross, Melbourne Central are consistently appeared among the top-ranked stations across multiple financial years, demonstrating their importance as key commuter and interchange hubs within Melbourne's metropolitan rail network.